In [1]:
!git clone https://github.com/minhoooo1/CatMAE.git /kaggle/working/CatMAE
!pip install timm opencv-python matplotlib scikit-image imageio scikit-learn visdom wandb gdown tqdm --quiet

Cloning into '/kaggle/working/CatMAE'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.ting objects:  76% (129/169)
remote: Compressing objects: 100% (121/121), done.
remote: Total 169 (delta 58), reused 149 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 21.57 MiB | 32.34 MiB/s, done.
Resolving deltas: 100% (58/58), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.9 MB/s eta 0:00:00:00:01


In [2]:
%%bash
mkdir -p /kaggle/working/checkpoints
gdown 1xWrpSxZy6d3r_XnsZmXvqM1XUReJ7v97 -O /kaggle/working/checkpoints/catmae_vits16.pth
gdown 1ksYZJPa2pZ-NYWjYKLh05-bt_A40Rhm7 -O /kaggle/working/checkpoints/catmae_vits8.pth

Downloading...
From (original): https://drive.google.com/uc?id=1xWrpSxZy6d3r_XnsZmXvqM1XUReJ7v97
From (redirected): https://drive.google.com/uc?id=1xWrpSxZy6d3r_XnsZmXvqM1XUReJ7v97&confirm=t&uuid=92f4bd46-aaca-4877-9ee8-b0c00dae9549
To: /kaggle/working/checkpoints/catmae_vits16.pth
100%|██████████| 113M/113M [00:00<00:00, 130MB/s]  
Downloading...
From (original): https://drive.google.com/uc?id=1ksYZJPa2pZ-NYWjYKLh05-bt_A40Rhm7
From (redirected): https://drive.google.com/uc?id=1ksYZJPa2pZ-NYWjYKLh05-bt_A40Rhm7&confirm=t&uuid=95f291e8-c550-4f40-be69-987becc581df
To: /kaggle/working/checkpoints/catmae_vits8.pth
100%|██████████| 133M/133M [00:03<00:00, 41.0MB/s] 


In [3]:
%%bash
cd /kaggle/working/CatMAE/downstream/davis2017-seg
mkdir -p data
# If you already have DAVIS-2017-trainval-480p somewhere, copy it in instead of these two lines:
wget -q -P ./data https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip
unzip -q ./data/DAVIS-2017-trainval-480p.zip -d ./data
cp -r data/DAVIS data/DAVIS_480_880
python convert_480_880.py   # resizes to 480x880 and writes davis_vallist_480_880.txt


###writing

/kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/JPEGImages/480p/bike-packing /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-packing
/kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/JPEGImages/480p/blackswan /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/blackswan
/kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/JPEGImages/480p/bmx-trees /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bmx-trees
/kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/JPEGImages/480p/breakdance /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/breakdance
/kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/JPEGImages/480p/camel /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/camel
/kaggle/working/CatMAE/downstream/davis2017-seg/data

100%|██████████| 90/90 [00:34<00:00,  2.62it/s]


In [4]:
!git clone https://github.com/davisvideochallenge/davis2017-evaluation.git /kaggle/working/davis2017-evaluation

Cloning into '/kaggle/working/davis2017-evaluation'...
remote: Enumerating objects: 6144, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 6144 (delta 2), reused 3 (delta 1), pack-reused 6132 (from 1)
Receiving objects: 100% (6144/6144), 18.49 MiB | 22.73 MiB/s, done.
Resolving deltas: 100% (578/578), done.


In [5]:
%%bash
set -e
cd /kaggle/working/CatMAE/downstream/davis2017-seg
python code/test_mae.py --model-type vits16 --resume /kaggle/working/checkpoints/catmae_vits16.pth \
  --gpu-id 0 --workers 0 \
  --save-path results/catmae_vits16/T07_K8_R18_V20 \
  --temperature 0.7 --topk 8 --radius 18 --videoLen 20 \
  --filelist davis_vallist_480_880.txt

Using GPU 0
Context Length: 20 Image Size: -1
Arguments Namespace(model_type='vits16', resume='/kaggle/working/checkpoints/catmae_vits16.pth', save_path='results/catmae_vits16/T07_K8_R18_V20', temperature=0.7, topk=8, radius=18.0, videoLen=20, cropSize=-1, filelist='davis_vallist_480_880.txt', gpu_id='0', batchSize=1, workers=0, manualSeed=777, visdom=False, visdom_server='localhost', head_depth=-1, remove_layers=['layer4'], no_l2=False, long_mem=[0], texture=False, round=False, norm_mask=False, finetune=0, pca_vis=False, device='cuda', imgSize=-1)
load [150/150] params
Total params: 21.67M
computing onehot lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-packing/00000_onehot.npy
computing resized lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-packing/00000_size30x55.npy
computing onehot lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-pac

0it [00:00, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
30it [09:07, 18.24s/it]


In [6]:
%%bash
set -e
cd /kaggle/working/CatMAE/downstream/davis2017-seg
python code/eval/convert_davis.py \
  --in_folder results/catmae_vits16/T07_K8_R18_V20/ \
  --out_folder results/catmae_vits16/T07_K8_R18_V20_out/ \
  --dataset data/DAVIS_480_880

total num is 30
0
13
4
17
23
3
16
9
12
22
7
20
27
5
14
28
2
15
24
1
11
19
26
8
18
25
6
10
21
29


In [7]:
%%bash
python /kaggle/working/davis2017-evaluation/evaluation_method.py \
  --davis_path /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880 \
  --set val --task semi-supervised \
  --results_path /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits16/T07_K8_R18_V20_out/

Evaluating sequences for the semi-supervised task...
Global results saved in /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits16/T07_K8_R18_V20_out/global_results-val.csv
Per-sequence results saved in /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits16/T07_K8_R18_V20_out/per-sequence_results-val.csv
--------------------------- Global results for val ---------------------------
 J&F-Mean  J-Mean  J-Recall  J-Decay   F-Mean  F-Recall  F-Decay
 0.625734 0.59693  0.714501 0.146983 0.654538  0.785712 0.211571

---------- Per sequence results for val ----------
            Sequence   J-Mean   F-Mean
      bike-packing_1 0.406219 0.572443
      bike-packing_2 0.679204 0.658555
         blackswan_1 0.876629 0.927494
         bmx-trees_1 0.283488 0.675288
         bmx-trees_2 0.537609 0.734585
        breakdance_1 0.676273 0.678081
             camel_1 0.703696 0.791708
    car-roundabout_1 0.864289 0.793371
        car-shadow_1 0.897774 0.937153
            

100%|██████████| 30/30 [01:18<00:00,  2.62s/it]


In [8]:
!ls /kaggle/working/CatMAE/downstream/davis2017-seg/code
!ls /kaggle/working/checkpoints
!ls /kaggle/working/davis2017-evaluation
!ls /kaggle/working/CatMAE/downstream/davis2017-seg/data

data  model.py	     __pycache__  test_mae.py  train.py
eval  models_mae.py  resnet.py	  test.py      utils
catmae_vits16.pth  catmae_vits8.pth
davis2017	       evaluation_method.py  pytest	results    setup.py
evaluation_codalab.py  LICENSE		     README.md	setup.cfg
DAVIS  DAVIS-2017-trainval-480p.zip  DAVIS_480_880


In [9]:
%%writefile /kaggle/working/patch_catmae_ram.py
import sys

root = sys.argv[1]  # path to .../downstream/davis2017-seg/code
tm = f"{root}/test_mae.py"
tu = f"{root}/utils/test_utils.py"

# ---------- test_mae.py ----------
s = open(tm).read()
if "feats_flat" in s:
    print("test_mae.py already patched")
else:
    old1 = "            keys, query = feats[:, :, key_indices], feats[:, :, n_context:]\n"
    new1 = "            feats_flat = feats.flatten(-2)   # [1, C, N, H*W]  (no 21x duplicated key bank)\n"
    assert old1 in s, "pattern 1 not found"
    s = s.replace(old1, new1)

    old2 = "            keys, query = keys.flatten(-2), query.flatten(-2)\n"
    new2 = "            query = feats_flat[:, :, n_context:]\n"
    assert old2 in s, "pattern 2 not found"
    s = s.replace(old2, new2)

    old3 = "Ws, Is = test_utils.mem_efficient_batched_affinity(query, keys, D, \n"
    new3 = "Ws, Is = test_utils.mem_efficient_batched_affinity(query, feats_flat, key_indices, D, \n"
    assert old3 in s, "pattern 3 not found"
    s = s.replace(old3, new3)

    old4 = "            torch.cuda.empty_cache()\n            print('******* Vid %s TOOK %s *******' % (vid_idx, time.time() - t_vid))\n"
    new4 = ("            del feats, feats_flat, query, Ws, Is, maps, imgs\n"
            "            gc.collect()\n"
            "            torch.cuda.empty_cache()\n"
            "            print('******* Vid %s TOOK %s *******' % (vid_idx, time.time() - t_vid))\n")
    assert old4 in s, "pattern 4 not found"
    s = s.replace(old4, new4)

    s = s.replace("import os\nimport time\n", "import os\nimport gc\nimport time\n", 1)
    open(tm, "w").write(s)
    print("patched test_mae.py")

# ---------- utils/test_utils.py ----------
u = open(tu).read()
if "key_indices" in u.split("def mem_efficient_batched_affinity")[1].split("def batched_affinity")[0]:
    print("test_utils.py already patched")
else:
    old = "def mem_efficient_batched_affinity(query, keys, mask, temperature, topk, long_mem, device):\n"
    new = "def mem_efficient_batched_affinity(query, feats_flat, key_indices, mask, temperature, topk, long_mem, device):\n"
    assert old in u, "pattern A not found"
    u = u.replace(old, new)

    old = "    for b in range(0, keys.shape[2], bsize):\n        _k, _q = keys[:, :, b:b+bsize].to(device), query[:, :, b:b+bsize].to(device)\n        w_s, i_s = [], []\n\n        for pb in range(0, _k.shape[-1], pbsize):"
    new = "    for b in range(0, key_indices.shape[0], bsize):\n        _k = feats_flat[:, :, key_indices[b:b+bsize]].to(device)   # gathered on the fly\n        _q = query[:, :, b:b+bsize].to(device)\n        w_s, i_s = [], []\n\n        for pb in range(0, _k.shape[-1], pbsize):"
    assert old in u, "pattern B not found"
    u = u.replace(old, new, 1)
    open(tu, "w").write(u)
    print("patched test_utils.py")

Writing /kaggle/working/patch_catmae_ram.py


In [10]:
!python /kaggle/working/patch_catmae_ram.py /kaggle/working/CatMAE/downstream/davis2017-seg/code

patched test_mae.py
patched test_utils.py


In [11]:
%%bash
set -e
cd /kaggle/working/CatMAE/downstream/davis2017-seg
python code/test_mae.py --model-type vits8 --resume /kaggle/working/checkpoints/catmae_vits8.pth \
  --gpu-id 1 --workers 0 \
  --save-path results/catmae_vits8/T05_K12_R32_V20 \
  --temperature 0.5 --topk 12 --radius 32 --videoLen 20 \
  --filelist davis_vallist_480_880.txt

Using GPU 1
Context Length: 20 Image Size: -1
Arguments Namespace(model_type='vits8', resume='/kaggle/working/checkpoints/catmae_vits8.pth', save_path='results/catmae_vits8/T05_K12_R32_V20', temperature=0.5, topk=12, radius=32.0, videoLen=20, cropSize=-1, filelist='davis_vallist_480_880.txt', gpu_id='1', batchSize=1, workers=0, manualSeed=777, visdom=False, visdom_server='localhost', head_depth=-1, remove_layers=['layer4'], no_l2=False, long_mem=[0], texture=False, round=False, norm_mask=False, finetune=0, pca_vis=False, device='cuda', imgSize=-1)
load [150/150] params
Total params: 21.67M
computing resized lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-packing/00000_size60x110.npy
computing resized lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bike-packing/00001_size60x110.npy
computing resized lbl for //kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880/Annotations/480p/bi

0it [00:00, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
30it [1:14:48, 149.63s/it]


In [12]:
%%bash
set -e
cd /kaggle/working/CatMAE/downstream/davis2017-seg
python code/eval/convert_davis.py \
  --in_folder results/catmae_vits8/T05_K12_R32_V20/ \
  --out_folder results/catmae_vits8/T05_K12_R32_V20_out/ \
  --dataset data/DAVIS_480_880

total num is 30
0
13
4
17
23
3
16
9
12
22
7
20
27
5
14
28
2
15
24
1
11
19
26
8
18
25
6
10
21
29


In [13]:
%%bash
python /kaggle/working/davis2017-evaluation/evaluation_method.py \
  --davis_path /kaggle/working/CatMAE/downstream/davis2017-seg/data/DAVIS_480_880 \
  --set val --task semi-supervised \
  --results_path /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits8/T05_K12_R32_V20_out/

Evaluating sequences for the semi-supervised task...
Global results saved in /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits8/T05_K12_R32_V20_out/global_results-val.csv
Per-sequence results saved in /kaggle/working/CatMAE/downstream/davis2017-seg/results/catmae_vits8/T05_K12_R32_V20_out/per-sequence_results-val.csv
--------------------------- Global results for val ---------------------------
 J&F-Mean   J-Mean  J-Recall  J-Decay   F-Mean  F-Recall  F-Decay
 0.706014 0.676102  0.804443 0.162543 0.735927  0.858857 0.212225

---------- Per sequence results for val ----------
            Sequence   J-Mean   F-Mean
      bike-packing_1 0.495111 0.682507
      bike-packing_2 0.765019 0.777902
         blackswan_1 0.941910 0.984817
         bmx-trees_1 0.284653 0.635440
         bmx-trees_2 0.589243 0.835540
        breakdance_1 0.683992 0.692815
             camel_1 0.728231 0.822281
    car-roundabout_1 0.891466 0.809116
        car-shadow_1 0.896953 0.873621
          

100%|██████████| 30/30 [01:18<00:00,  2.61s/it]


In [14]:
%%bash
R=/kaggle/working/CatMAE/downstream/davis2017-seg/results
mkdir -p /kaggle/working/github_upload/results
cp $R/catmae_vits16/T07_K8_R18_V20_out/global_results-val.csv        /kaggle/working/github_upload/results/vits16_global_results-val.csv
cp $R/catmae_vits16/T07_K8_R18_V20_out/per-sequence_results-val.csv  /kaggle/working/github_upload/results/vits16_per-sequence_results-val.csv
cp $R/catmae_vits8/T05_K12_R32_V20_out/global_results-val.csv        /kaggle/working/github_upload/results/vits8_global_results-val.csv
cp $R/catmae_vits8/T05_K12_R32_V20_out/per-sequence_results-val.csv  /kaggle/working/github_upload/results/vits8_per-sequence_results-val.csv
ls -la /kaggle/working/github_upload/results

total 24
drwxr-xr-x 2 root root 4096 Oct 20 22:06 .
drwxr-xr-x 3 root root 4096 Oct 20 22:06 ..
-rw-r--r-- 1 root root   99 Oct 20 22:06 vits16_global_results-val.csv
-rw-r--r-- 1 root root 1487 Oct 20 22:06 vits16_per-sequence_results-val.csv
-rw-r--r-- 1 root root   99 Oct 20 22:06 vits8_global_results-val.csv
-rw-r--r-- 1 root root 1487 Oct 20 22:06 vits8_per-sequence_results-val.csv


In [15]:
import shutil
shutil.make_archive('/kaggle/working/github_upload', 'zip', '/kaggle/working', 'github_upload')

'/kaggle/working/github_upload.zip'